In [1]:
# ============================================================================
# FINAL HYBRID OFFLINE BUILD — paragraph-aware chunking + dense FAISS index
# + precomputed BM25 inverted index. Self-contained, Unicode-safe, full corpus.
# Progress printed at each ~10% of embedding, with live ETA.
# Saves: index.faiss, index_meta.json, index_texts.json, bm25.npz, bm25_vocab.json
# ============================================================================
import json, re, math, time
from pathlib import Path
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

# ---- paths & config ----
ROOT = Path(r"C:\Users\User\Desktop\שנה ד\סמסטר ח\מעבדה בניתוח והצגת נתונים 00940295\Project_Part_A\Part_B")
ENTRIES_DIR   = ROOT / "data" / "Wikipedia Entries"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM  = 384
BATCH      = 256          # embed batch size
PROGRESS_EVERY = 0.10     # print embed progress each ~10%
# chunking
TARGET_WORDS, HARD_MAX, OVERLAP_SENTS = 150, 200, 1
# BM25
BM25_K1, BM25_B = 1.5, 0.75

assert ENTRIES_DIR.is_dir(), f"Corpus not found: {ENTRIES_DIR}"
_T0 = time.perf_counter()
def _t(): return time.perf_counter() - _T0

# ---- tokenizer & sentence splitter ----
_TOK = re.compile(r"[a-z0-9]+")
def tokenize(s): return _TOK.findall(s.lower())
def split_sentences(s):
    return [p for p in re.split(r"(?<=[.!?])\s+", s.strip()) if p]

# ---- balanced paragraph-aware chunker (pack to TARGET, sentence overlap, title prefix) ----
def chunk_page(title, content):
    title = (title or "").strip(); content = (content or "").strip()
    if not content:
        return [title] if title else [""]
    paras = [p.strip() for p in content.split("\n\n") if p.strip()] or [content]
    bins, cur, cur_n, carry = [], [], 0, []
    def flush():
        nonlocal cur, cur_n, carry
        if cur:
            bins.append(" ".join(cur))
            sents = split_sentences(" ".join(cur))
            carry = sents[-OVERLAP_SENTS:] if OVERLAP_SENTS else []
            cur = list(carry); cur_n = sum(len(s.split()) for s in carry)
    for para in paras:
        pw = para.split()
        if len(pw) > HARD_MAX:                     # oversized paragraph: window alone
            flush()
            step = max(1, TARGET_WORDS - 30)
            for s in range(0, len(pw), step):
                bins.append(" ".join(pw[s:s + TARGET_WORDS]))
                if s + TARGET_WORDS >= len(pw): break
            cur, cur_n, carry = [], 0, []
            continue
        if cur and cur_n + len(pw) > TARGET_WORDS: flush()
        cur.append(para); cur_n += len(pw)
    if cur and cur != carry: bins.append(" ".join(cur))
    pref = (title + "\n\n") if title else ""
    return [(pref + b).strip() for b in bins] if bins else [title]

# ===========================================================================
print(f"[1/5] Reading corpus ...  ({_t():.0f}s)")
records = []
for p in sorted(ENTRIES_DIR.glob("*.json")):
    rec = json.loads(p.read_text(encoding="utf-8"))
    pid = rec.get("page_id", p.stem)
    pid = int(pid) if str(pid).strip().lstrip("-").isdigit() else int(p.stem)
    records.append((pid, rec.get("title", ""), rec.get("content", "")))
print(f"      pages: {len(records)}")

print(f"[2/5] Chunking (paragraph-aware) ...  ({_t():.0f}s)")
texts, page_ids, chunk_ids = [], [], []
for pid, title, content in records:
    for cid, ch in enumerate(chunk_page(title, content)):
        page_ids.append(pid); chunk_ids.append(cid); texts.append(ch)
N = len(texts)
print(f"      chunks: {N}  (avg {N/max(1,len(records)):.2f} chunks/page)")

print(f"[3/5] Embedding {N} chunks ...  ({_t():.0f}s)")
model = SentenceTransformer(MODEL_NAME); model.max_seq_length = 256
vecs = np.zeros((N, EMBED_DIM), dtype=np.float32)
emb_t0 = time.perf_counter(); next_mark = PROGRESS_EVERY; done = 0
for start in range(0, N, BATCH):
    chunk = texts[start:start + BATCH]
    v = model.encode(chunk, batch_size=BATCH, show_progress_bar=False,
                     convert_to_numpy=True, normalize_embeddings=True)
    vecs[start:start + len(chunk)] = v
    done += len(chunk)
    frac = done / N
    if frac >= next_mark or done == N:
        el = time.perf_counter() - emb_t0
        rate = done / el if el > 0 else 0.0
        if done <= BATCH:
            print(f"      [embed] {done:,}/{N:,} ({frac*100:4.1f}%) | "
                  f"{rate:5.1f} ch/s | elapsed {el:5.1f}s | ETA warming up")
        else:
            eta = (N - done) / rate if rate > 0 else 0.0
            print(f"      [embed] {done:,}/{N:,} ({frac*100:4.1f}%) | "
                  f"{rate:5.1f} ch/s | elapsed {el:5.1f}s | ETA ~{eta/60:4.1f}m")
        next_mark += PROGRESS_EVERY
print(f"      embedded in {time.perf_counter()-emb_t0:.1f}s")

print(f"[4/5] Building FAISS + BM25 inverted index ...  ({_t():.0f}s)")
# --- dense index ---
index = faiss.IndexFlatIP(EMBED_DIM)
if N: index.add(np.ascontiguousarray(vecs, dtype=np.float32))
# --- BM25 (weights precomputed: independent of query) ---
toks = [tokenize(t) for t in texts]
doc_len = np.array([len(t) for t in toks], dtype=np.int32)
avgdl = float(doc_len.mean()) if N else 0.0
df = {}
for t in toks:
    for term in set(t): df[term] = df.get(term, 0) + 1
vocab = sorted(df); vid = {w: i for i, w in enumerate(vocab)}
idf = np.array([math.log((N - df[w] + 0.5) / (df[w] + 0.5) + 1) for w in vocab],
               dtype=np.float32)
postings = {}   # term_id -> list[(doc, weight)]
for d, t in enumerate(toks):
    tf = {}
    for term in t: tf[term] = tf.get(term, 0) + 1
    for term, f in tf.items():
        denom = f + BM25_K1 * (1 - BM25_B + BM25_B * doc_len[d] / avgdl)
        w = idf[vid[term]] * (f * (BM25_K1 + 1)) / denom
        postings.setdefault(vid[term], []).append((d, float(w)))
term_ptr = np.zeros(len(vocab) + 1, dtype=np.int64)
pd, pw = [], []
for ti in range(len(vocab)):
    for d, w in postings.get(ti, []): pd.append(d); pw.append(w)
    term_ptr[ti + 1] = len(pd)
posting_docs = np.array(pd, dtype=np.int32)
posting_wts  = np.array(pw, dtype=np.float32)
print(f"      vocab: {len(vocab)} terms | postings: {len(posting_docs):,}")

print(f"[5/5] Saving artifacts ...  ({_t():.0f}s)")
(ARTIFACTS_DIR / "index.faiss").write_bytes(faiss.serialize_index(index).tobytes())
(ARTIFACTS_DIR / "index_meta.json").write_text(json.dumps({
    "page_ids": page_ids, "chunk_ids": chunk_ids, "model": MODEL_NAME,
    "dim": EMBED_DIM, "num_vectors": N, "num_pages": len(set(page_ids)),
    "chunker": "paragraph", "target_words": TARGET_WORDS,
}, indent=2), encoding="utf-8")
(ARTIFACTS_DIR / "index_texts.json").write_text(
    json.dumps(texts, ensure_ascii=False), encoding="utf-8")
np.savez(ARTIFACTS_DIR / "bm25.npz",
         term_ptr=term_ptr, posting_docs=posting_docs, posting_wts=posting_wts,
         doc_len=doc_len, k1=np.float32(BM25_K1), b=np.float32(BM25_B),
         avgdl=np.float32(avgdl), num_docs=np.int32(N))
(ARTIFACTS_DIR / "bm25_vocab.json").write_text(
    json.dumps(vocab, ensure_ascii=False), encoding="utf-8")

# ---- verify ----
re_idx = faiss.deserialize_index(
    np.frombuffer((ARTIFACTS_DIR / "index.faiss").read_bytes(), dtype=np.uint8).copy())
print(f"\nDONE in {_t():.1f}s")
print(f"  pages={len(set(page_ids))}  chunks={N}  vocab={len(vocab)}")
for fn in ("index.faiss", "index_meta.json", "index_texts.json", "bm25.npz", "bm25_vocab.json"):
    fp = ARTIFACTS_DIR / fn
    print(f"  {fn:18s} {fp.stat().st_size/1024/1024:8.2f} MB  exists={fp.exists()}")
print(f"  faiss reload ntotal={re_idx.ntotal} (matches={re_idx.ntotal==N})")

[1/5] Reading corpus ...  (0s)
      pages: 27074
[2/5] Chunking (paragraph-aware) ...  (4s)
      chunks: 628751  (avg 23.22 chunks/page)
[3/5] Embedding 628751 chunks ...  (18s)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      [embed] 62,976/628,751 (10.0%) |  28.5 ch/s | elapsed 2207.4s | ETA ~330.5m
      [embed] 125,952/628,751 (20.0%) |  27.8 ch/s | elapsed 4529.6s | ETA ~301.4m
      [embed] 188,672/628,751 (30.0%) |  26.9 ch/s | elapsed 7005.0s | ETA ~272.3m
      [embed] 251,648/628,751 (40.0%) |  27.2 ch/s | elapsed 9259.9s | ETA ~231.3m
      [embed] 314,624/628,751 (50.0%) |  27.3 ch/s | elapsed 11531.9s | ETA ~191.9m
      [embed] 377,344/628,751 (60.0%) |  27.4 ch/s | elapsed 13777.9s | ETA ~153.0m
      [embed] 440,320/628,751 (70.0%) |  27.7 ch/s | elapsed 15900.8s | ETA ~113.4m
      [embed] 503,040/628,751 (80.0%) |  28.0 ch/s | elapsed 17956.4s | ETA ~74.8m
      [embed] 566,016/628,751 (90.0%) |  28.1 ch/s | elapsed 20128.5s | ETA ~37.2m
      [embed] 628,751/628,751 (100.0%) |  28.0 ch/s | elapsed 22444.9s | ETA ~ 0.0m
      embedded in 22444.9s
[4/5] Building FAISS + BM25 inverted index ...  (22467s)
      vocab: 490282 terms | postings: 50,985,007
[5/5] Saving artifacts ...  (22899

In [ ]:
# ============================================================================
# CELL 2 — PAGE + BM25 BUILD (converter). Runs after Cell 1 (which built the
# chunk index.faiss). Self-contained: reads data/ directly, embeds one vector
# per page, builds page-level BM25, writes the artifacts that retrieve.py loads.
# Outputs -> artifacts/: page_vecs.npy, page_meta.json, page_texts.json,
#                        bm25.npz, bm25_vocab.json
# ============================================================================
import json, math, re, time
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

ROOT = Path(r"C:\Users\User\Desktop\שנה ד\סמסטר ח\מעבדה בניתוח והצגת נתונים 00940295\Project_Part_A\Part_B")
ENTRIES_DIR   = ROOT / "data" / "Wikipedia Entries"
ARTIFACTS_DIR = ROOT / "artifacts"; ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM, BATCH = 384, 256
PAGE_WORD_CAP = 400                 # title+content cap (MiniLM truncates ~256 tokens anyway)
BM25_K1, BM25_B = 1.5, 0.75
_TOK = re.compile(r"[a-z0-9]+")
def tokenize(s): return _TOK.findall(s.lower())

assert ENTRIES_DIR.is_dir(), f"Corpus not found: {ENTRIES_DIR}"
T0 = time.perf_counter()

# ---- 1. one capped title+content text per page ----
print("[1/4] Reading corpus -> one text per page ...")
page_ids, texts = [], []
for p in sorted(ENTRIES_DIR.glob("*.json")):
    rec = json.loads(p.read_text(encoding="utf-8"))
    pid = rec.get("page_id", p.stem)
    pid = int(pid) if str(pid).strip().lstrip("-").isdigit() else int(p.stem)
    title = str(rec.get("title", "")).strip()
    content = " ".join(str(rec.get("content", "")).split()[:PAGE_WORD_CAP])
    text = f"{title}\n\n{content}".strip() if title else content
    page_ids.append(pid); texts.append(text if text else title)
P = len(texts); print(f"      pages: {P}")

# ---- 2. page-level dense vectors ----
print(f"[2/4] Embedding {P} page vectors ...")
model = SentenceTransformer(MODEL_NAME); model.max_seq_length = 256
pvecs = np.zeros((P, EMBED_DIM), dtype=np.float32)
t0 = time.perf_counter()
for s in range(0, P, BATCH):
    ch = texts[s:s+BATCH]
    pvecs[s:s+len(ch)] = model.encode(ch, batch_size=BATCH, show_progress_bar=False,
                                      convert_to_numpy=True, normalize_embeddings=True)
    if (s // BATCH) % 20 == 0:
        print(f"      ...{min(s+BATCH, P):,}/{P:,}", flush=True)
print(f"      embedded in {time.perf_counter()-t0:.1f}s")

# ---- 3. page-level BM25 (precomputed posting weights, CSR-by-term) ----
print("[3/4] Building page-level BM25 ...")
toks = [tokenize(t) for t in texts]
doc_len = np.array([len(t) for t in toks], dtype=np.float64)
avgdl = float(doc_len.mean()) if P else 0.0
vocab, df, page_tf = {}, {}, []
for t in toks:
    tf = {}
    for w in t:
        tid = vocab.setdefault(w, len(vocab)); tf[tid] = tf.get(tid, 0) + 1
    page_tf.append(tf)
    for tid in tf: df[tid] = df.get(tid, 0) + 1
idf = np.zeros(len(vocab))
for tid, d in df.items():
    idf[tid] = math.log((P - d + 0.5) / (d + 0.5) + 1.0)          # +1 keeps idf >= 0
per_term = [[] for _ in range(len(vocab))]
for prow, tf in enumerate(page_tf):
    c = BM25_K1 * (1 - BM25_B + BM25_B * (doc_len[prow] / avgdl if avgdl else 0.0))
    for tid, f in tf.items():
        per_term[tid].append((prow, idf[tid] * (f * (BM25_K1 + 1)) / (f + c)))
term_ptr = np.zeros(len(vocab) + 1, dtype=np.int64); pages, wts = [], []
for tid, post in enumerate(per_term):
    for prow, w in post: pages.append(prow); wts.append(w)
    term_ptr[tid + 1] = len(pages)
print(f"      vocab: {len(vocab)} | postings: {len(pages):,}")

# ---- 4. save (formats match index.load_hybrid / retrieve.py) ----
print("[4/4] Saving artifacts ...")
np.save(ARTIFACTS_DIR / "page_vecs.npy", pvecs)
(ARTIFACTS_DIR / "page_meta.json").write_text(json.dumps({"page_ids": page_ids}), encoding="utf-8")
(ARTIFACTS_DIR / "page_texts.json").write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")
np.savez(ARTIFACTS_DIR / "bm25.npz", term_ptr=term_ptr,
         post_pages=np.asarray(pages, dtype=np.int32),
         post_weights=np.asarray(wts, dtype=np.float32))
(ARTIFACTS_DIR / "bm25_vocab.json").write_text(json.dumps(vocab, ensure_ascii=False), encoding="utf-8")

print(f"\nDONE in {time.perf_counter()-T0:.1f}s")
for fn in ("page_vecs.npy","page_meta.json","page_texts.json","bm25.npz","bm25_vocab.json"):
    fp = ARTIFACTS_DIR / fn
    print(f"  {fn:18s} {fp.stat().st_size/1e6:8.2f} MB  exists={fp.exists()}")
print(f"  page_vecs={pvecs.shape}  L2-norm[0]={np.linalg.norm(pvecs[0]):.4f}  vocab={len(vocab)}")